# Paired comparison of Score106 and Score249


In [ ]:
from pathlib import Path
for required_source in ("score106.py", "metrics.py"):
    required_path = Path.cwd().parent / "sources" / required_source
    if not required_path.is_file():
        raise FileNotFoundError(f"Required analysis module not found: {required_path}")

from pathlib import Path
import os
import importlib.util
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits

BOOTSTRAPS = int(os.environ.get("CHD_BOOTSTRAP_N", "1000"))
N_JOBS = min(24, BOOTSTRAPS)

PACKAGE = Path.cwd().parent
PROJECT = Path(os.environ.get("CHD_OUTPUT_ROOT", "../outputs")).resolve()
RUNS = ["reviewer_full249_score_5fold", "reviewer_raw106_score"]
COHORT = "prs_raw106_complete"
FEATURE = "Clinical_plus_PRS_plus_NMR_score"

snapshots = []
for run in RUNS:
    os.environ.update(
        CHD_OUTPUT_ROOT=str(PROJECT),
        CHD_RUN_ID=run,
        CHD_COHORT_ID=COHORT,
        CHD_SENSITIVITY_OUTPUT_DIR=str(PROJECT / "runs" / run),
    )
    get_ipython().run_line_magic(
        "run", '"' + str((Path.cwd() / '00_shared_config.ipynb')) + '"'
    )
    df_train, df_val, df_test, feature_sets, manifest = \
        validate_pipeline_artifacts(True)
    record = load_all_risk_records(
        {FEATURE: feature_sets[FEATURE]}, ["Cox PH"]
    )[(MAIN_SCENARIO_NAME, FEATURE, "Cox PH")]
    snapshots.append((df_train.copy(), df_test.copy(), record, manifest))

full249, score106 = snapshots

for a, b in [(full249[0], score106[0]), (full249[1], score106[1])]:
    if (
        eid_digest(a) != eid_digest(b)
        or not np.array_equal(a.Event.to_numpy(), b.Event.to_numpy())
        or not np.allclose(
            a.Time.to_numpy(float), b.Time.to_numpy(float),
            rtol=0, atol=1e-12
        )
    ):
        raise RuntimeError("The paired runs differ in participant order or outcomes.")

METRICS_SOURCE = PACKAGE / "sources" / "metrics.py"
if not METRICS_SOURCE.is_file():
    raise FileNotFoundError(f"Required analysis module not found: {METRICS_SOURCE}")
spec = importlib.util.spec_from_file_location(
    "review_metrics", METRICS_SOURCE
)
M = importlib.util.module_from_spec(spec)
spec.loader.exec_module(M)
import sys
try:
    from joblib.externals import cloudpickle
except ImportError:
    import cloudpickle
sys.modules["review_metrics"] = M
if hasattr(cloudpickle, "register_pickle_by_value"):
    cloudpickle.register_pickle_by_value(M)

calibrators = [
    M.fit_absolute_risk_calibrator(s[0], s[2]["train_risk"], run)
    for s, run in zip(snapshots, RUNS)
]

OUT = (
    PROJECT / "runs" / RUNS[1] / "04_results"
    / "score106_vs_score249_valid5"
)
OUT.mkdir(exist_ok=True, parents=True)

train_min = full249[0][["Time", "Event"]].reset_index(drop=True)
test_min = full249[1][["Time", "Event"]].reset_index(drop=True)

risk_new = np.asarray(score106[2]["test_risk"], dtype=float)
risk_ref = np.asarray(full249[2]["test_risk"], dtype=float)

def uno_bootstrap_block(index_block, train_frame, eval_frame,
                        risk_new, risk_ref, horizon):
    values = []
    with threadpool_limits(limits=1):
        for idx in index_block:
            boot = eval_frame.iloc[idx].reset_index(drop=True)
            if boot["Event"].nunique() < 2:
                continue
            try:
                values.append(
                    M.uno_c(train_frame, boot, risk_new[idx], horizon)
                    - M.uno_c(train_frame, boot, risk_ref[idx], horizon)
                )
            except Exception:
                continue
    return values

def nri_bootstrap_block(index_block, eval_frame, pred_ref,
                        pred_new, horizon, cutpoints):
    values = {"NRI_Categorical": [], "NRI_Continuous": [], "IDI": []}
    with threadpool_limits(limits=1):
        for idx in index_block:
            boot = eval_frame.iloc[idx].reset_index(drop=True)
            try:
                result = M.ipcw_nri_idi_once(
                    boot, pred_ref[idx], pred_new[idx], horizon,
                    cutpoints=cutpoints
                )
                for key in values:
                    values[key].append(result[key])
            except Exception:
                continue
    return values

def parallel_chunks(index_matrix):
    return [
        index_matrix[chunk]
        for chunk in np.array_split(np.arange(len(index_matrix)), N_JOBS)
        if len(chunk)
    ]

rows = []
for horizon in [5.0, 10.0]:
    M.horizon_support_audit(train_min, test_min, [horizon])

    rng = np.random.default_rng(M.BOOTSTRAP_SEED)
    n = len(test_min)
    bootstrap_indices = rng.integers(
        0, n, size=(BOOTSTRAPS, n), dtype=np.int64
    )
    chunks = parallel_chunks(bootstrap_indices)

    point_new = M.uno_c(train_min, test_min, risk_new, horizon)
    point_ref = M.uno_c(train_min, test_min, risk_ref, horizon)
    uno_parts = Parallel(
        n_jobs=N_JOBS, backend="loky", batch_size=1,
        pre_dispatch=N_JOBS, verbose=10
    )(
        delayed(uno_bootstrap_block)(
            chunk, train_min, test_min, risk_new, risk_ref, horizon
        )
        for chunk in chunks
    )
    uno_values = [x for part in uno_parts for x in part]
    uno_lo, uno_hi = M.quantile_ci(uno_values)

    ibs_objects = [
        M.subject_level_ipcw_ibs(
            s[0][["Time", "Event"]].reset_index(drop=True),
            s[1][["Time", "Event"]].reset_index(drop=True),
            cal, s[2]["test_risk"], horizon
        )
        for s, cal in zip(snapshots, calibrators)
    ]
    paired_subject_ibs = (
        np.asarray(ibs_objects[1]["Subject_IBS"])
        - np.asarray(ibs_objects[0]["Subject_IBS"])
    )
    ibs_values = np.mean(
        paired_subject_ibs[bootstrap_indices], axis=1
    )
    ibs_lo, ibs_hi = M.quantile_ci(ibs_values)

    pred_ref = M.predict_absolute_risk(
        calibrators[0], risk_ref, horizon
    )
    pred_new = M.predict_absolute_risk(
        calibrators[1], risk_new, horizon
    )
    nri_point = M.ipcw_nri_idi_once(
        test_min, pred_ref, pred_new, horizon,
        cutpoints=M.NRI_CUTPOINTS
    )
    nri_parts = Parallel(
        n_jobs=N_JOBS, backend="loky", batch_size=1,
        pre_dispatch=N_JOBS, verbose=10
    )(
        delayed(nri_bootstrap_block)(
            chunk, test_min, pred_ref, pred_new,
            horizon, M.NRI_CUTPOINTS
        )
        for chunk in chunks
    )
    nri_values = {
        key: [value for part in nri_parts for value in part[key]]
        for key in ["NRI_Categorical", "NRI_Continuous", "IDI"]
    }

    row = {
        "Horizon": horizon,
        "New": "NMR_score106",
        "Reference": "NMR_score249_5fold",
        "Uno_C_New": point_new,
        "Uno_C_Reference": point_ref,
        "Delta_Uno_C": point_new - point_ref,
        "Delta_Uno_C_Lower95": uno_lo,
        "Delta_Uno_C_Upper95": uno_hi,
        "Bootstrap_N_Uno_Used": len(uno_values),
        "IBS_New": ibs_objects[1]["IBS"],
        "IBS_Reference": ibs_objects[0]["IBS"],
        "Delta_IBS": float(np.mean(paired_subject_ibs)),
        "Delta_IBS_Lower95": ibs_lo,
        "Delta_IBS_Upper95": ibs_hi,
        "Bootstrap_N_IBS_Used": len(ibs_values),
        **nri_point,
        "Bootstrap_N_NRI_IDI_Used": min(
            len(v) for v in nri_values.values()
        ),
    }

    for key, values in nri_values.items():
        lower, upper = M.quantile_ci(values)
        row[f"{key}_Lower95"] = lower
        row[f"{key}_Upper95"] = upper

    rows.append(row)

result = pd.DataFrame(rows)
output_path = OUT / "paired_score106_minus_score249.csv"
result.to_csv(output_path, index=False)
print(result.to_string(index=False))
print("Saved:", output_path)
print(f"Bootstrap workers: {N_JOBS}; replicates per horizon: {BOOTSTRAPS}")
